# Bước 1: Khởi tạo môi trường & Import thư viện

In [2]:
import pandas as pd
import os
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

# 1. Đọc dữ liệu gốc từ thư mục raw
df = pd.read_csv('../data/raw/HR_Analytics.csv')

# 2. Dọn dẹp sơ bộ (Xóa các cột ID không có giá trị dự đoán nếu có)
cols_to_drop = ['EmpID', 'EmployeeCount', 'EmployeeNumber', 'Over18', 'StandardHours']
cols_to_drop = [c for c in cols_to_drop if c in df.columns]
df = df.drop(columns=cols_to_drop)

# 3. Xử lý Missing Values (Điền khuyết)
for col in df.columns:
    if df[col].isnull().sum() > 0:
        if df[col].dtype == 'object':
            df[col].fillna(df[col].mode()[0], inplace=True)
        else:
            df[col].fillna(df[col].median(), inplace=True)

# 4. Xác định và mã hóa biến mục tiêu (Target)
target_col = 'Attrition'
if target_col not in df.columns:
    possible_targets = [c for c in df.columns if c.lower() in ['attrition', 'left', 'turnover']]
    if possible_targets: target_col = possible_targets[0]

# Đổi chữ 'Yes'/'No' thành 1/0
if df[target_col].dtype == 'object':
    df[target_col] = df[target_col].apply(lambda x: 1 if str(x).lower() in ['yes', '1', 'true'] else 0)

# 5. Mã hóa One-Hot Encoding cho các cột chữ (Biến phân loại)
cat_cols = df.select_dtypes(include=['object']).columns
df_encoded = pd.get_dummies(df, columns=cat_cols, drop_first=True)

# 6. Cắt dữ liệu thành Train và Test
X = df_encoded.drop(target_col, axis=1)
y = df_encoded[target_col]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 7. LƯU FILE VÀO ĐÚNG THƯ MỤC (Đây là phần sửa lỗi cho bạn)
# Lệnh này đảm bảo thư mục data/processed chắc chắn được tạo ra
os.makedirs('../data/processed', exist_ok=True)

# Bắt đầu lưu 4 file
X_train.to_csv('../data/processed/X_train.csv', index=False)
X_test.to_csv('../data/processed/X_test.csv', index=False)
y_train.to_csv('../data/processed/y_train.csv', index=False)
y_test.to_csv('../data/processed/y_test.csv', index=False)

print("✅ THÀNH CÔNG RỰC RỠ: Máy tính đã tạo xong file X_test.csv và lưu đúng vào thư mục data/processed!")

✅ THÀNH CÔNG RỰC RỠ: Máy tính đã tạo xong file X_test.csv và lưu đúng vào thư mục data/processed!


# Bước 2: Load dữ liệu và kiểm tra

In [2]:
# Gọi hàm load dữ liệu từ config
df = load_raw_data(config_path='configs/params.yaml')

print(f"Kích thước ban đầu: {df.shape}")

# Xóa các cột định danh không có giá trị phân tích (như EmpID, EmployeeCount, StandardHours, Over18 nếu có)
cols_to_drop = ['EmpID', 'EmployeeCount', 'StandardHours', 'Over18', 'EmployeeNumber']
df.drop(columns=[col for col in cols_to_drop if col in df.columns], inplace=True)

Đã load dữ liệu thành công! Kích thước dữ liệu: (1480, 38)
Kích thước ban đầu: (1480, 38)


# Bước 3: Làm sạch dữ liệu

In [3]:
missing_values = df.isnull().sum().sum()
duplicates = df.duplicated().sum()

print(f"Số giá trị bị thiếu: {missing_values}")
print(f"Số dòng trùng lặp: {duplicates}")

if missing_values > 0:
    df.dropna(inplace=True)
if duplicates > 0:
    df.drop_duplicates(inplace=True)

# TẠO 2 BẢN SAO CHO 2 MỤC ĐÍCH KHÁC NHAU
df_ml = df.copy()    # Dùng cho Machine Learning (chỉ chứa số)
df_rules = df.copy() # Dùng cho Association Rules (chứa Text/Category)

Số giá trị bị thiếu: 57
Số dòng trùng lặp: 7


# Bước 4: Xử lý dữ liệu cho Machine Learning (df_ml)
* Yêu cầu: Đưa tất cả về dạng số để chạy Random Forest / XGBoost.

In [5]:
# 1. Mã hóa biến mục tiêu Attrition (Nghỉ việc) thành 1 và 0
if 'Attrition' in df_ml.columns:
    df_ml['Attrition'] = df_ml['Attrition'].map({'Yes': 1, 'No': 0})

# 2. Mã hóa One-Hot Encoding cho các biến phân loại (Categorical)
categorical_cols = df_ml.select_dtypes(include=['object']).columns
df_ml = pd.get_dummies(df_ml, columns=categorical_cols, drop_first=True)

# Chuyển đổi True/False thành 1/0
for col in df_ml.columns:
    if df_ml[col].dtype == bool:
        df_ml[col] = df_ml[col].astype(int)

display(df_ml.head(3))

,Age,Attrition,DailyRate,DistanceFromHome,Education,EnvironmentSatisfaction,HourlyRate,JobInvolvement,JobLevel,JobSatisfaction,...,JobRole_Research Director,JobRole_Research Scientist,JobRole_Sales Executive,JobRole_Sales Representative,MaritalStatus_Married,MaritalStatus_Single,SalarySlab_15k+,SalarySlab_5k-10k,SalarySlab_Upto 5k,OverTime_Yes
0,18,1,230,3,3,3,54,3,1,3,...,0,0,0,0,0,1,0,0,1,0
1,18,0,812,10,3,4,69,2,1,3,...,0,0,0,1,0,1,0,0,1,0
2,18,1,1306,5,3,2,69,3,1,2,...,0,0,0,1,0,1,0,0,1,1


# Bước 5: Rời rạc hóa cho Khai phá luật kết hợp (df_rules)
* Yêu cầu: Chuyển các biến số (như Tuổi, Lương) thành các khoảng (Thấp/Trung bình/Cao) dưới dạng Text để Apriori hiểu được.

In [6]:
labels_3 = ['Thấp', 'Trung bình', 'Cao']

# 1. Tuổi (Age)
if 'Age' in df_rules.columns:
    df_rules['Age'] = pd.qcut(df_rules['Age'], q=3, labels=['Trẻ', 'Trung niên', 'Lớn tuổi'])
    df_rules['Age'] = 'Age_' + df_rules['Age'].astype(str)

# 2. Thu nhập hàng tháng (MonthlyIncome)
if 'MonthlyIncome' in df_rules.columns:
    df_rules['MonthlyIncome'] = pd.qcut(df_rules['MonthlyIncome'], q=3, labels=labels_3)
    df_rules['MonthlyIncome'] = 'Income_' + df_rules['MonthlyIncome'].astype(str)

# 3. Khoảng cách từ nhà đến công ty (DistanceFromHome)
if 'DistanceFromHome' in df_rules.columns:
    df_rules['DistanceFromHome'] = pd.cut(df_rules['DistanceFromHome'], bins=3, labels=['Gần', 'Vừa', 'Xa'])
    df_rules['DistanceFromHome'] = 'Distance_' + df_rules['DistanceFromHome'].astype(str)

# 4. Số năm làm việc tại công ty (YearsAtCompany)
if 'YearsAtCompany' in df_rules.columns:
    df_rules['YearsAtCompany'] = pd.cut(df_rules['YearsAtCompany'], bins=[-1, 3, 7, 50], labels=['Mới', 'Gắn bó', 'Lão làng'])
    df_rules['YearsAtCompany'] = 'Tenure_' + df_rules['YearsAtCompany'].astype(str)

# 5. Gắn tiền tố cho các cột Categorical có sẵn để phân biệt rõ ràng
df_rules['Attrition'] = df_rules['Attrition'].apply(lambda x: 'Status_Leave' if x == 'Yes' else 'Status_Stay')

for col in ['Department', 'JobRole', 'BusinessTravel']:
    if col in df_rules.columns:
        df_rules[col] = col + '_' + df_rules[col].astype(str)

display(df_rules[['Age', 'MonthlyIncome', 'DistanceFromHome', 'YearsAtCompany', 'Attrition']].head(3))

,Age,MonthlyIncome,DistanceFromHome,YearsAtCompany,Attrition
0,Age_Trẻ,Income_Thấp,Distance_Gần,Tenure_Mới,Status_Leave
1,Age_Trẻ,Income_Thấp,Distance_Gần,Tenure_Mới,Status_Stay
2,Age_Trẻ,Income_Thấp,Distance_Gần,Tenure_Mới,Status_Leave


# Bước 6: Lưu dữ liệu đã xử lý vào `data/processed`

In [7]:
processed_dir = 'data/processed'
os.makedirs(processed_dir, exist_ok=True)

# Lưu 2 file dữ liệu
df_ml.to_csv(f'{processed_dir}/hr_ml_encoded.csv', index=False)
df_rules.to_csv(f'{processed_dir}/hr_rules_binned.csv', index=False)

print("Đã lưu thành công:")
print(f"1. {processed_dir}/hr_ml_encoded.csv (Dùng cho mô hình dự đoán)")
print(f"2. {processed_dir}/hr_rules_binned.csv (Dùng cho khai phá luật)")

Đã lưu thành công:
1. data/processed/hr_ml_encoded.csv (Dùng cho mô hình dự đoán)
2. data/processed/hr_rules_binned.csv (Dùng cho khai phá luật)
